# [재실행 전용 · 2026-10-04 회계 수정] 평가 전용 진단 2차

학습 셀 없이 저장된 모델만 불러와, 논문에서 '회계 수정 전 수치'로 남아 있던 평가를 다시 계산한다.
- §23: 4.4절 v4 다양성 진단(ρ 분포, β별 평균 ρ, top-10 순환 종목 수)
- §24: 노출 맞춤 벤치마크(배포 정책의 낮은 MDD가 현금 비중 때문인지)
- §25: 4.4절 저턴오버 아티팩트 진단(전반부/후반부 수익, 턴오버 추이, 종목 회전)
- §19: 6.10절 블록 순열검정

위에서부터 **순서대로 전부 실행**하면 된다. 결과 CSV는 `data/crypto/`에 저장된다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
import warnings
import glob
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['axes.unicode_minus'] = False

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data'
CRYPTO_DIR = DATA_DIR / 'crypto'
FEAT_DIR   = CRYPTO_DIR / 'features_5m'
MODEL_DIR  = DRIVE_ROOT / 'models'
POOL_DIR   = MODEL_DIR / 'crypto_hrl_pool'

RL_EMB_H5      = CRYPTO_DIR / 'crypto_rl_embeddings.h5'
ORIG_PPO_PATH  = MODEL_DIR / 'crypto_ppo_portfolio.zip'   # crypto_rl_trading_colab.ipynb 산출물(있으면 비교에 포함)

EMB_DIM    = 64
TC         = 0.001    # 거래비용 (Binance taker fee 기준)
BUCKET_SEC = 1800     # 30분 버킷

RL_TRAIN_START = '2025-01-01'
RL_TRAIN_END   = '2025-12-31'
RL_TEST_START  = '2026-01-01'
RL_TEST_END    = '2099-12-31'   # 열린 구간

# --- Stage A/B/C 하이퍼파라미터 ---
N_LABELS              = 5          # 레짐 라벨 수 (bear/pullback/sideways/rally/bull)
RISK_THRESHOLD        = 0.2        # EarnHFT θ — 상하위 라벨 경계 분위수 폭
LOWPASS_WINDOW        = 48         # 레짐 분할 저역통과 윈도우 = 1일(30분×48)
POOL_BETAS            = [-90, -30, 30, 90]   # Stage B 선호도 스윕 (논문 [-90,-10,30,100] 근사)
POOL_TIMESTEPS_EACH   = 100_000    # 풀 에이전트 1개당 학습 스텝 (원본 300k와 총량 비슷하게)
ROUTE_BUCKETS         = 48         # Stage C 라우팅 단위 = 1일
MACRO_WINDOW           = 96        # 라우터 상태에 포함할 최근 버킷 수 (=2일)
ROUTER_TOTAL_TIMESTEPS = 20_000    # 라우터(=일 단위) 학습 스텝

assert RL_EMB_H5.exists(), 'crypto_rl_embeddings.h5 없음 — crypto_hybrid_model_colab.ipynb 먼저 실행'
print('경로 확인 완료')


In [ ]:
print('crypto_rl_embeddings.h5 로드 중...')
with h5py.File(RL_EMB_H5, 'r') as f:
    symbols_raw = f['symbols'][:]
    ts_raw      = f['timestamps'][:]     # epoch seconds (UTC)
    embs_all    = f['embeddings'][:]     # (N, 64)

symbols_list = [s.decode() for s in symbols_raw]
ts_list      = ts_raw.astype(np.int64).tolist()

# (symbol, 30분 버킷) → 임베딩 인덱스
emb_index  = {}
sym_ts_set = {}   # 심볼별로 임베딩 추출에 실제 쓰인 초 단위 timestamp 집합 (수익률 정확 매칭용)
for i, (s, t) in enumerate(zip(symbols_list, ts_list)):
    emb_index[(s, t // BUCKET_SEC)] = i
    sym_ts_set.setdefault(s, set()).add(t)

symbols  = sorted(set(symbols_list))
N_ASSETS = len(symbols)
print(f'임베딩: {embs_all.shape} | 심볼: {N_ASSETS}개 | 인덱스: {len(emb_index):,}개')


In [ ]:
print('features_5m parquet에서 RetTarget_6b 로드 중...')
files = sorted(glob.glob(str(FEAT_DIR / '*.parquet')))
assert files, f'피처 parquet 없음: {FEAT_DIR}'

rl_start_ts = pd.Timestamp(RL_TRAIN_START, tz='UTC')
ret_index   = {}

for fp in files:
    sym = Path(fp).stem
    valid_ts = sym_ts_set.get(sym)
    if not valid_ts:
        continue
    df = pd.read_parquet(fp, columns=['datetime', 'RetTarget_6b'])
    df = df[df['datetime'] >= rl_start_ts]
    epoch = df['datetime'].values.astype('int64') // 10**9   # ts_list와 동일 단위(초)
    y     = df['RetTarget_6b'].values
    # 임베딩 추출에 실제 쓰인 timestamp만 매칭 — 같은 버킷 내 다른 5분봉과 섞이지 않도록
    for t, r in zip(epoch, y):
        t = int(t)
        if t in valid_ts and not np.isnan(r):
            ret_index[(sym, t // BUCKET_SEC)] = float(r)

all_buckets = sorted(set(t // BUCKET_SEC for t in ts_list))
print(f'수익률 인덱스: {len(ret_index):,}개 | 전체 30분 버킷: {len(all_buckets):,}개')


In [ ]:
# [2026-10-04 회계 수정] 가격 변동 후 실제 보유 비중(drift)·거래 가능 종목 마스크.
# 이전 버전은 수익 반영 뒤에도 직전 '목표' 비중을 보유 비중으로 저장해, 가격 변동으로 틀어진 비중을
# 되돌리는 재조정 거래(등가중 포함)의 비용이 0으로 계산됐다.
def _drift_weights(w, rets):
    """수익 반영 뒤 실제 보유 비중 w_i(1+r_i)/(1+wᵀr). 합<1이면 나머지는 현금(수익률 0)으로 남는다."""
    w = np.asarray(w, dtype=np.float64)
    g = 1.0 + float(np.dot(w, rets))
    if g <= 1e-12:
        return w.astype(np.float32)
    return (w * (1.0 + np.asarray(rets, dtype=np.float64)) / g).astype(np.float32)


def _mask_untradable(w, tradable, renorm=True):
    """수익률 데이터가 없는(상장 전·상폐 후) 종목 비중을 0으로. renorm=False면 빠진 몫은 현금으로 남는다."""
    w = np.where(tradable, np.asarray(w, dtype=np.float64), 0.0)
    s = w.sum()
    return (w / s if renorm and s > 1e-12 else w).astype(np.float32)


class CryptoPortfolioEnv(gym.Env):
    """
    크립토 44종목 개별 자산 기반 포트폴리오 운용 환경 (클러스터링 없음)

    State:  종목별 임베딩(N×64) + 현재 비중(N) → N*64+N dim
    Action: 종목별 목표 포트폴리오 비중 (N-dim, Softmax 정규화)
    Reward: 30분 포트폴리오 수익률(RetTarget_6b, 이미 forward-looking) − 거래비용
    스텝 = 30분 버킷(BUCKET_SEC) 단위 — emb_index/ret_index 모두 (symbol, bucket) 키.
    """
    metadata = {'render_modes': []}

    def __init__(self, buckets, symbols, emb_index, embs_all, ret_index,
                 emb_dim=64, tc=0.001):
        super().__init__()
        self.buckets    = buckets
        self.symbols    = symbols
        self.n_assets   = len(symbols)
        self.emb_index  = emb_index
        self.embs_all   = embs_all
        self.ret_index  = ret_index
        self.emb_dim    = emb_dim
        self.tc         = tc

        obs_dim = self.n_assets * emb_dim + self.n_assets
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Box(-3.0, 3.0, shape=(self.n_assets,), dtype=np.float32)

    def _asset_emb(self, bucket):
        """종목별 임베딩 — 해당 버킷에 데이터 없으면 0-vector(상장 전/상폐 등)"""
        embs = []
        for sym in self.symbols:
            idx = self.emb_index.get((sym, bucket))
            embs.append(self.embs_all[idx] if idx is not None else np.zeros(self.emb_dim, dtype=np.float32))
        return np.concatenate(embs)  # (N*64,)

    def _asset_ret(self, bucket):
        """종목별 30분 수익률 — 없으면 0(그 자산은 보유해도 손익 없음으로 취급)"""
        return np.array([self.ret_index.get((sym, bucket), 0.0) for sym in self.symbols], dtype=np.float32)

    def _tradable(self, bucket):
        """해당 버킷에 수익률 데이터가 있는(거래 가능한) 종목 마스크"""
        return np.array([(sym, bucket) in self.ret_index for sym in self.symbols])

    def _get_obs(self):
        b = self.buckets[self.t]
        return np.concatenate([self._asset_emb(b), self.weights]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t               = 0
        self.weights         = _mask_untradable(np.ones(self.n_assets, dtype=np.float32),
                                          self._tradable(self.buckets[0]))  # 거래 가능 종목 등가중에서 시작
        self.portfolio_value = 1.0
        self.value_history   = [1.0]
        self.turnover_history = []
        return self._get_obs(), {}

    def step(self, action):
        # Softmax로 비중 정규화
        action  = np.exp(action - action.max())
        action  = action / action.sum()
        action  = _mask_untradable(action, self._tradable(self.buckets[self.t]))  # 거래 불가 종목 제외

        # 거래비용
        turnover = float(np.abs(action - self.weights).sum())  # 비중 변화량(진단용)
        tc_cost  = self.tc * turnover

        # RetTarget_6b는 현재 버킷 자체에 forward 수익률이 이미 인코딩되어 있음
        b          = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret   = float((action * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = _drift_weights(action, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t                += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)

        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret, 'turnover': turnover, 'tc_cost': tc_cost}
        return self._get_obs(), reward, done, False, info


In [ ]:
def bucket_to_ts(b):
    return pd.Timestamp(b * BUCKET_SEC, unit='s', tz='UTC')

def bucket_to_date(b):
    return bucket_to_ts(b).strftime('%Y-%m-%d')

train_buckets = [b for b in all_buckets if RL_TRAIN_START <= bucket_to_date(b) <= RL_TRAIN_END]
test_buckets  = [b for b in all_buckets if RL_TEST_START  <= bucket_to_date(b) <= RL_TEST_END]
print(f'학습 기간: {bucket_to_ts(train_buckets[0])} ~ {bucket_to_ts(train_buckets[-1])} ({len(train_buckets):,}스텝)')
print(f'테스트 기간: {bucket_to_ts(test_buckets[0])} ~ {bucket_to_ts(test_buckets[-1])} ({len(test_buckets):,}스텝)')

env_kwargs = dict(
    symbols=symbols, emb_index=emb_index, embs_all=embs_all,
    ret_index=ret_index, emb_dim=EMB_DIM, tc=TC,
)

test_env = CryptoPortfolioEnv(test_buckets, **env_kwargs)

print('\n환경 유효성 검사...')
check_env(CryptoPortfolioEnv(train_buckets[:100], **env_kwargs), warn=True)
print('OK')


In [ ]:
# crypto/market_segmentation.py 이식 (Colab은 저장소를 마운트하지 않으므로 동일 코드를 셀에 포함)
REGIME_NAMES = {1: 'bear', 2: 'pullback', 3: 'sideways', 4: 'rally', 5: 'bull'}
_DTW_MAX_LEN = 200


@dataclass
class MarketSegments:
    labels: pd.Series
    chunks: pd.DataFrame


def _zscore(x):
    std = x.std()
    return (x - x.mean()) / std if std > 1e-12 else np.zeros_like(x)


def _downsample(x, max_len=_DTW_MAX_LEN):
    if len(x) <= max_len:
        return x
    idx = np.linspace(0, len(x) - 1, max_len).round().astype(int)
    return x[idx]


def _dtw_distance(a, b):
    a, b = _downsample(a), _downsample(b)
    n, m = len(a), len(b)
    cost = np.full((n + 1, m + 1), np.inf)
    cost[0, 0] = 0.0
    for i in range(1, n + 1):
        ai = a[i - 1]
        row, prev = cost[i], cost[i - 1]
        for j in range(1, m + 1):
            row[j] = abs(ai - b[j - 1]) + min(prev[j], row[j - 1], prev[j - 1])
    return float(cost[n, m])


def _slope(x):
    if len(x) < 2:
        return 0.0
    t = np.arange(len(x), dtype=np.float64)
    a, _ = np.polyfit(t, x, 1)
    return float(a)


def _find_extrema_indices(x):
    d = np.diff(x)
    sign = np.sign(d)
    sign[sign == 0] = 1
    change = np.where(np.diff(sign) != 0)[0] + 1
    idx = np.concatenate(([0], change, [len(x) - 1]))
    return np.unique(idx)


def _merge_pass(x, bounds, merge_quantile):
    if len(bounds) <= 2:
        return bounds, False
    seg_ranges = list(zip(bounds[:-1], bounds[1:]))
    slopes = [_slope(x[a:b + 1]) for a, b in seg_ranges]
    if len(seg_ranges) < 2:
        return bounds, False
    slope_diffs = [abs(slopes[i + 1] - slopes[i]) for i in range(len(slopes) - 1)]
    dtw_dists = [
        _dtw_distance(
            _zscore(x[seg_ranges[i][0]:seg_ranges[i][1] + 1]),
            _zscore(x[seg_ranges[i + 1][0]:seg_ranges[i + 1][1] + 1]),
        )
        for i in range(len(seg_ranges) - 1)
    ]
    slope_thresh = np.quantile(slope_diffs, merge_quantile)
    dtw_thresh = np.quantile(dtw_dists, merge_quantile)

    new_bounds = [bounds[0]]
    merged_any = False
    i = 0
    while i < len(seg_ranges):
        can_merge = (
            i < len(seg_ranges) - 1
            and slope_diffs[i] <= slope_thresh
            and dtw_dists[i] <= dtw_thresh
        )
        if can_merge:
            new_bounds.append(seg_ranges[i + 1][1])
            merged_any = True
            i += 2
        else:
            new_bounds.append(seg_ranges[i][1])
            i += 1
    return new_bounds, merged_any


def _absorb_short_segments(x, bounds, min_length):
    while len(bounds) > 2:
        seg_ranges = list(zip(bounds[:-1], bounds[1:]))
        lengths = [b - a + 1 for a, b in seg_ranges]
        shortest = int(np.argmin(lengths))
        if lengths[shortest] >= min_length:
            break
        slopes = [_slope(x[a:b + 1]) for a, b in seg_ranges]
        left_diff = abs(slopes[shortest] - slopes[shortest - 1]) if shortest > 0 else np.inf
        right_diff = (
            abs(slopes[shortest] - slopes[shortest + 1]) if shortest < len(seg_ranges) - 1 else np.inf
        )
        drop_idx = shortest if left_diff <= right_diff else shortest + 1
        bounds = bounds[:drop_idx] + bounds[drop_idx + 1:]
    return bounds


def segment_and_label(series, n_labels=5, risk_threshold=0.2, lowpass_window=48,
                       merge_quantile=0.35, max_merge_iters=50, min_segment_length=None):
    """가격 시계열을 레짐 구간으로 분할하고 라벨을 부여한다 (EarnHFT Algorithm 3 이식).
    로그가격(buy&hold net curve의 로그) 기준으로 계산해 가격 스케일 왜곡을 방지한다."""
    x_raw = series.to_numpy(dtype=np.float64)
    if len(x_raw) < 3:
        raise ValueError('series가 너무 짧습니다 (최소 3개 포인트 필요)')
    if np.any(x_raw <= 0):
        raise ValueError('series는 로그 변환을 위해 모두 양수여야 합니다')

    x_log = np.log(x_raw)
    x_smooth = pd.Series(x_log).rolling(lowpass_window, min_periods=1, center=True).mean().to_numpy()

    bounds = _find_extrema_indices(x_smooth).tolist()
    for _ in range(max_merge_iters):
        bounds, merged = _merge_pass(x_smooth, bounds, merge_quantile)
        if not merged or len(bounds) <= 2:
            break
    bounds = _absorb_short_segments(x_log, bounds, min_segment_length or lowpass_window)

    seg_ranges = list(zip(bounds[:-1], bounds[1:]))
    slopes = np.array([_slope(x_log[a:b + 1]) for a, b in seg_ranges])

    theta = risk_threshold
    H = np.quantile(slopes, 1 - theta / 2)
    L = np.quantile(slopes, theta / 2)

    labels = np.empty(len(slopes), dtype=np.int64)
    for i, s in enumerate(slopes):
        if s > H:
            labels[i] = n_labels
        elif s < L:
            labels[i] = 1
        elif H > L:
            frac = (s - L) / (H - L)
            labels[i] = int(np.clip(2 + frac * (n_labels - 2), 2, n_labels - 1))
        else:
            labels[i] = (n_labels + 1) // 2

    per_point_labels = np.empty(len(x_raw), dtype=np.int64)
    for (a, b), lbl in zip(seg_ranges, labels):
        per_point_labels[a:b + 1] = lbl

    labels_series = pd.Series(per_point_labels, index=series.index, name='regime_label')

    chunks = pd.DataFrame({
        'start_idx': [a for a, _ in seg_ranges],
        'end_idx': [b for _, b in seg_ranges],
        'length': [b - a + 1 for a, b in seg_ranges],
        'slope': slopes,
        'label': labels,
    })
    chunks['regime'] = chunks['label'].map(REGIME_NAMES) if n_labels == 5 else chunks['label'].astype(str)
    return MarketSegments(labels=labels_series, chunks=chunks)


def chunk_sampling_priority(chunks, beta, risk_threshold=0.2):
    """EarnHFT Eq.4 이식 — chunk 기울기(r)에 대한 β-지수가중 샘플링 우선순위(합 1)."""
    r = chunks['slope'].to_numpy(dtype=np.float64)
    theta = risk_threshold
    Ht = np.quantile(r, 1 - theta / 2)
    Lt = np.quantile(r, theta / 2)

    std = r.std() if r.std() > 1e-12 else 1.0
    h = max(0.9 * std * len(r) ** (-1 / 5), 1e-8)
    diffs = (r[:, None] - r[None, :]) / h
    pdf = np.exp(-0.5 * diffs ** 2).sum(axis=1) / (len(r) * h * np.sqrt(2 * np.pi))
    pdf = np.maximum(pdf, 1e-12)

    tail_mask = (r >= Ht) | (r <= Lt)
    log_w = beta * r - np.where(tail_mask, 0.0, np.log(pdf))
    log_w -= log_w.max()
    weights = np.exp(log_w)
    return weights / weights.sum()


print('market_segmentation 함수 정의 완료')


In [ ]:
# 44종목 등가중 NAV 곡선 구성 (거래비용 제외 — 시장 자체의 추세 파악용)
eq_ret_per_bucket = np.array([
    np.mean([ret_index.get((sym, b), 0.0) for sym in symbols])
    for b in train_buckets
])
eq_nav = pd.Series(np.cumprod(1 + eq_ret_per_bucket), index=range(len(train_buckets)))

seg = segment_and_label(
    eq_nav, n_labels=N_LABELS, risk_threshold=RISK_THRESHOLD,
    lowpass_window=LOWPASS_WINDOW,
)
print(f"구간 수: {len(seg.chunks)} (평균 길이 {seg.chunks['length'].mean():.0f}버킷 ≈ {seg.chunks['length'].mean()/48:.1f}일)")
print(seg.chunks['regime'].value_counts())

fig, ax = plt.subplots(figsize=(14, 4))
colors = {'bear': 'firebrick', 'pullback': 'lightsalmon', 'sideways': 'lightgray', 'rally': 'lightgreen', 'bull': 'darkgreen'}
for _, row in seg.chunks.iterrows():
    ax.axvspan(row['start_idx'], row['end_idx'], color=colors[row['regime']], alpha=0.5)
ax.plot(eq_nav.values, color='black', lw=1)
ax.set_title('44종목 등가중 NAV + Stage A 레짐 분할 (2025 학습 구간)')
ax.set_xlabel('버킷 인덱스 (train_buckets 내 위치)')
plt.tight_layout()
plt.show()


In [ ]:
TOP_K = 10   # 종목 유니버스(44) 중 매 스텝 보유 가능한 최대 종목 수 (FreQuant top-|G| 이식)

class SparseGatedCryptoPortfolioEnv(CryptoPortfolioEnv):
    """
    FreQuant(top-K 희소 선택) + DeepClair(전용 리스크게이트 ρ) 결합 저수준 환경.

    Action: (N+1,)-dim — 앞 N개는 종목별 confidence logit(FreQuant Eq.13-14의 h_i 역할),
            마지막 1개는 리스크자산 총 노출 게이트 로짓(DeepClair의 ρ_t 역할).
    변환:   top-K confidence logit만 골라 그 안에서 softmax(FreQuant Eq.14) →
            sigmoid(게이트)를 곱해 최종 종목별 비중. 나머지는 암묵적 현금(수익률 0).
    관측·NAV 회계는 CryptoPortfolioEnv와 동일. step_with_weights()는 목표 비중을 직접
    받아 NAV를 갱신하는 보조 메서드 — §13-G 관성 스무딩에서 (top-K 변환을 다시 태우지 않고)
    이미 블렌딩된 비중을 그대로 반영하기 위해 필요하다.
    """

    def __init__(self, *args, top_k=TOP_K, **kwargs):
        self.top_k = top_k
        super().__init__(*args, **kwargs)
        self.action_space = spaces.Box(-3.0, 3.0, shape=(self.n_assets + 1,), dtype=np.float32)

    def action_to_weights(self, action):
        action = np.asarray(action, dtype=np.float64)
        asset_logits, gate_logit = action[:-1], action[-1]
        asset_logits = np.where(self._tradable(self.buckets[self.t]), asset_logits, -np.inf)  # 거래 불가 종목 제외
        k = min(self.top_k, self.n_assets)
        top_idx = np.argpartition(asset_logits, -k)[-k:]
        sel = asset_logits[top_idx]
        sel = np.exp(sel - sel.max())
        sel = sel / sel.sum()
        risky_w = np.zeros(self.n_assets, dtype=np.float32)
        risky_w[top_idx] = sel
        rho = float(1.0 / (1.0 + np.exp(-gate_logit)))
        return (rho * risky_w).astype(np.float32), rho, top_idx

    def reset(self, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        self.rho_history = []
        self.selection_history = []
        return obs, info

    def step_with_weights(self, target_w, extra_info=None):
        target_w = _mask_untradable(target_w, self._tradable(self.buckets[self.t]), renorm=False)  # 빠진 몫은 현금
        turnover = float(np.abs(target_w - self.weights).sum())
        tc_cost  = self.tc * turnover
        b          = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret   = float((target_w * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = _drift_weights(target_w, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t                += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)
        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret,
                'turnover': turnover, 'tc_cost': tc_cost}
        if extra_info:
            info.update(extra_info)
            if 'rho' in extra_info:
                self.rho_history.append(extra_info['rho'])
            if 'selected' in extra_info:
                self.selection_history.append(extra_info['selected'])
        return self._get_obs(), reward, done, False, info

    def step(self, action):
        target_w, rho, top_idx = self.action_to_weights(action)
        return self.step_with_weights(target_w, extra_info={'rho': rho, 'selected': top_idx.tolist()})


# 스모크 테스트
_sg_env = SparseGatedCryptoPortfolioEnv(test_buckets[:200], top_k=TOP_K, **env_kwargs)
check_env(_sg_env, warn=True)
obs, _ = _sg_env.reset()
a = np.random.randn(_sg_env.n_assets + 1).astype(np.float32)
obs, r, done, _, info = _sg_env.step(a)
print(f"action space: {_sg_env.action_space.shape} | 선택된 종목 수: {len(info['selected'])} | "
      f"rho: {info['rho']:.3f} | reward: {r:.5f}")

In [ ]:
class ChunkedSparseGatedEnv(SparseGatedCryptoPortfolioEnv):
    """§5의 ChunkedCryptoPortfolioEnv와 동일한 β-우선순위 청크 샘플링, 저수준 환경만 v4로 교체."""

    def __init__(self, all_buckets, chunks, beta, risk_threshold=0.2, rng=None, top_k=TOP_K, **env_kwargs):
        self.all_buckets = all_buckets
        self.chunks      = chunks.reset_index(drop=True)
        self.beta        = beta
        self.priority    = chunk_sampling_priority(self.chunks, beta, risk_threshold)
        self.rng         = rng or np.random.default_rng()
        super().__init__(buckets=all_buckets[:2], top_k=top_k, **env_kwargs)  # reset()에서 실제 구간으로 교체됨

    def reset(self, seed=None, options=None):
        idx = self.rng.choice(len(self.chunks), p=self.priority)
        row = self.chunks.iloc[idx]
        self.buckets = self.all_buckets[int(row['start_idx']):int(row['end_idx']) + 1]
        return super().reset(seed=seed, options=options)


# 스모크 테스트
_test_env_v4 = ChunkedSparseGatedEnv(
    all_buckets=train_buckets, chunks=seg.chunks, beta=90, rng=np.random.default_rng(0), **env_kwargs
)
for _ in range(3):
    _test_env_v4.reset()
    print(f"β=90 샘플 구간: {len(_test_env_v4.buckets)}버킷 "
          f"({bucket_to_date(_test_env_v4.buckets[0])} ~ {bucket_to_date(_test_env_v4.buckets[-1])})")

In [ ]:
class MacroRouterEnvV4(gym.Env):
    """§6 MacroRouterEnv와 동일 로직, 저수준 환경만 SparseGatedCryptoPortfolioEnv로 교체.

    episode_progress_mode: 관측값 마지막 성분(t/len(buckets), '에피소드 내 진행률').
      - 'real'(기본값, §13-C/§13-D 학습에 사용): 실제 진행률을 그대로 넘김 — 원 EarnHFT
        라우터 관측 설계를 그대로 재현하지만, 실거래에는 '에피소드 끝'이 없어
        policy_v4.py는 이 값을 상수 0.5로 근사한다(라이브가 학습 때 없던 입력 분포를
        받는 구조적 불일치, 모듈 docstring 한계 3).
      - 'constant': 학습 중에도 항상 0.5로 고정 — 라이브가 실제로 보게 될 입력 분포와
        학습 분포를 일치시킨 '매칭' 라우터(§13-I)용.
    """
    metadata = {'render_modes': []}

    def __init__(self, buckets, pool_models, route_buckets=48, macro_window=96, top_k=TOP_K,
                 episode_progress_mode='real', **env_kwargs):
        super().__init__()
        self.pool_models           = list(pool_models)
        self.route_buckets         = route_buckets
        self.macro_window          = macro_window
        self.episode_progress_mode = episode_progress_mode
        self.inner           = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=top_k, **env_kwargs)
        self.action_space      = spaces.Discrete(len(self.pool_models))
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(macro_window + 4,), dtype=np.float32)
        self._nav_history = []

    def _macro_obs(self):
        hist = self._nav_history[-self.macro_window:]
        if len(hist) < self.macro_window:
            hist = [1.0] * (self.macro_window - len(hist)) + hist
        recent = np.array(hist, dtype=np.float32)
        ret = np.diff(recent) if len(recent) > 1 else np.zeros(1, dtype=np.float32)
        if self.episode_progress_mode == 'constant':
            episode_progress = 0.5
        else:
            episode_progress = self.inner.t / max(len(self.inner.buckets) - 1, 1)
        stats = np.array([
            recent[-1] - recent[0],
            float(ret.std()),
            float(ret.mean()),
            episode_progress,
        ], dtype=np.float32)
        return np.concatenate([recent, stats]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.inner.reset(seed=seed)
        self._nav_history = []
        return self._macro_obs(), {}

    def step(self, action):
        model = self.pool_models[int(action)]
        block_reward = 0.0
        done_inner = False
        for _ in range(self.route_buckets):
            obs = self.inner._get_obs()
            a, _ = model.predict(obs, deterministic=True)
            _, r, done_inner, _, info = self.inner.step(a)
            block_reward += r
            self._nav_history.append(info['portfolio_value'])
            if done_inner:
                break
        done = done_inner or self.inner.t >= len(self.inner.buckets) - 1
        info = {'portfolio_value': self.inner.portfolio_value}
        return self._macro_obs(), block_reward, done, False, info


# 스모크 테스트 — pool_models_v4가 아직 없는 세션(§13-C를 건너뛰고 §13-로더만 나중에 실행하는
# 복구 경로)에서도 이 셀이 클래스 정의만은 항상 성공하도록 방어.
if 'pool_models_v4' in globals() and pool_models_v4:
    _router_probe_v4 = MacroRouterEnvV4(
        buckets=train_buckets[:500], pool_models=list(pool_models_v4.values()),
        route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, top_k=TOP_K, **env_kwargs,
    )
    obs, _ = _router_probe_v4.reset()
    print('[v4] router obs shape:', obs.shape)
    obs, r, done, _, info = _router_probe_v4.step(0)
    print('[v4] 1스텝 후 reward:', r, 'portfolio_value:', info['portfolio_value'])
else:
    print('MacroRouterEnvV4 클래스 정의 완료 (pool_models_v4 없음 — 스모크 테스트 생략, §13-로더 실행 후 계속 진행)')


In [ ]:
# [v4] 진단 재실행용 로더 — 세션이 끊겨도 저장된 모델만 불러와 이어서 백테스트 가능
# (§13-C 학습 셀을 건너뛴 경로에서는 POOL_DIR_V4가 아직 없으므로 여기서 재정의 — 경로만 가리킬 뿐 idempotent)
POOL_DIR_V4 = MODEL_DIR / 'crypto_hrl_pool_v4'
if 'pool_models_v4' not in globals() or not pool_models_v4:
    pool_models_v4 = {b: PPO.load(str(POOL_DIR_V4 / f'agent_beta_{b}.zip')) for b in POOL_BETAS}
    print('[v4] 풀 에이전트 로드 완료:', list(pool_models_v4.keys()))
if 'router_v4' not in globals():
    router_v4 = DQN.load(str(MODEL_DIR / 'crypto_hrl_router_v4.zip'))
    print('[v4] 라우터 로드 완료')

# 버그 수정: 이 로더만 실행하고 §13-B(셀 46)를 건너뛴 세션에서는 ChunkedSparseGatedEnv가
# 정의되지 않아 §13-G 다양성 진단(β별 단독 프로브)이 NameError로 죽는다 — 여기서 idempotent하게 복구.
if 'ChunkedSparseGatedEnv' not in globals():
    class ChunkedSparseGatedEnv(SparseGatedCryptoPortfolioEnv):
        """§13-B(셀 46)와 동일 정의 — 재실행 경로에서 누락됐을 때만 여기서 복구."""

        def __init__(self, all_buckets, chunks, beta, risk_threshold=0.2, rng=None, top_k=TOP_K, **env_kwargs):
            self.all_buckets = all_buckets
            self.chunks      = chunks.reset_index(drop=True)
            self.beta        = beta
            self.priority    = chunk_sampling_priority(self.chunks, beta, risk_threshold)
            self.rng         = rng or np.random.default_rng()
            super().__init__(buckets=all_buckets[:2], top_k=top_k, **env_kwargs)  # reset()에서 실제 구간으로 교체됨

        def reset(self, seed=None, options=None):
            idx = self.rng.choice(len(self.chunks), p=self.priority)
            row = self.chunks.iloc[idx]
            self.buckets = self.all_buckets[int(row['start_idx']):int(row['end_idx']) + 1]
            return super().reset(seed=seed, options=options)

    print('[v4] ChunkedSparseGatedEnv 재정의 완료 (§13-B 건너뛴 세션 대비 — §13-G 진단에 필요)')


## 진단 (원본 노트북 §23과 동일)

ρ 분포, β별 평균 ρ를 세 가지 방식((a) 원래 방식: 학습 청크 1개 / (b) 학습 청크 20개 / (c) 테스트 전 구간 단독)으로,
top-K 종목 순환 수를 라우터+풀과 β별 단독으로 잰다. 논문 4.4절 수치(ρ 표준편차 0.068, β별 평균 ρ 0.109~0.545, 33/44종목)와 대조한다.


In [ ]:
# §23 — v4 다양성 진단 재실행 (자기 완결형)
from collections import Counter

# 0) §13-E(셀 54)의 라우터+풀 테스트 실행을 이 셀 안에서 재현 (세션에 없을 때만)
if 'router_test_env_v4' not in globals():
    print('[§23] router_test_env_v4 없음 → 테스트 구간 라우터+풀 실행 재현 중...')
    router_test_env_v4 = MacroRouterEnvV4(
        buckets=test_buckets, pool_models=[pool_models_v4[b] for b in POOL_BETAS],
        route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, top_k=TOP_K, **env_kwargs,
    )
    _obs, _ = router_test_env_v4.reset()
    _done = False
    while not _done:
        _a, _ = router_v4.predict(_obs, deterministic=True)
        _obs, _, _done, _, _ = router_test_env_v4.step(_a)

rows = []

# 1) 라우터+풀 실행(스무딩 없음, 테스트 구간) 중 ρ 분포
rho_arr = np.array(router_test_env_v4.inner.rho_history)
print(f'[1] 라우터+풀 ρ — 평균 {rho_arr.mean():.3f} | 표준편차 {rho_arr.std():.3f} | '
      f'최소 {rho_arr.min():.3f} | 최대 {rho_arr.max():.3f} | n={len(rho_arr)}')
rows.append({'diag': 'router_pool_test_rho', 'beta': 'router', 'mean_rho': rho_arr.mean(),
             'std_rho': rho_arr.std(), 'n': len(rho_arr)})

# 2a) 원래 방식 — 학습 구간 청크 1개(rng=999), 비교용
# 2b) 보강 — 학습 구간 청크 20개 평균
# 2c) 보강 — 테스트 전 구간 단독 실행(스무딩 없음)
def run_chunk_probe(beta, rng_seed, n_chunks):
    rng = np.random.default_rng(rng_seed)
    rhos = []
    for _ in range(n_chunks):
        probe = ChunkedSparseGatedEnv(
            all_buckets=train_buckets, chunks=seg.chunks, beta=beta,
            risk_threshold=RISK_THRESHOLD, rng=rng, top_k=TOP_K, **env_kwargs,
        )
        obs, _ = probe.reset()
        done = False
        while not done:
            a, _ = pool_models_v4[beta].predict(obs, deterministic=True)
            obs, _, done, _, _ = probe.step(a)
        rhos.extend(probe.rho_history)
    return np.array(rhos)

def run_test_solo(beta):
    env = SparseGatedCryptoPortfolioEnv(buckets=test_buckets, top_k=TOP_K, **env_kwargs)
    obs, _ = env.reset()
    done = False
    while not done:
        a, _ = pool_models_v4[beta].predict(obs, deterministic=True)
        obs, _, done, _, _ = env.step(a)
    return np.array(env.rho_history), env.selection_history

print('\n[2] β별 평균 ρ — (a) 원래 방식: 학습 청크 1개 | (b) 학습 청크 20개 | (c) 테스트 전 구간 단독')
solo_selections = {}
for beta in POOL_BETAS:
    r_a = run_chunk_probe(beta, 999, 1)
    r_b = run_chunk_probe(beta, 999, 20)
    r_c, sel_c = run_test_solo(beta)
    solo_selections[beta] = sel_c
    print(f'  β={beta:>4d}: (a) {r_a.mean():.3f} | (b) {r_b.mean():.3f} ± {r_b.std():.3f} | '
          f'(c) {r_c.mean():.3f} ± {r_c.std():.3f}')
    rows += [
        {'diag': 'beta_mean_rho_train_1chunk', 'beta': beta, 'mean_rho': r_a.mean(), 'std_rho': r_a.std(), 'n': len(r_a)},
        {'diag': 'beta_mean_rho_train_20chunks', 'beta': beta, 'mean_rho': r_b.mean(), 'std_rho': r_b.std(), 'n': len(r_b)},
        {'diag': 'beta_mean_rho_test_solo', 'beta': beta, 'mean_rho': r_c.mean(), 'std_rho': r_c.std(), 'n': len(r_c)},
    ]

# 3) 종목 선택 다양성 — top-K 안에 한 번이라도 뽑힌 종목 수
cnt = Counter()
for sel in router_test_env_v4.inner.selection_history:
    cnt.update(sel)
print(f'\n[3] 라우터+풀(테스트): top-{TOP_K}에 한 번이라도 뽑힌 종목 {len(cnt)} / {N_ASSETS}')
rows.append({'diag': 'router_pool_test_unique_topk', 'beta': 'router', 'unique_symbols': len(cnt), 'n_assets': N_ASSETS})
for beta in POOL_BETAS:
    c = Counter()
    for sel in solo_selections[beta]:
        c.update(sel)
    print(f'    β={beta:>4d} 단독(테스트): {len(c)} / {N_ASSETS}')
    rows.append({'diag': 'beta_solo_test_unique_topk', 'beta': beta, 'unique_symbols': len(c), 'n_assets': N_ASSETS})

diag_df = pd.DataFrame(rows)
diag_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v4_diversity_diag.csv', index=False, encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_v4_diversity_diag.csv')
print('▶ 논문 4.4절 수치(ρ 표준편차 0.068 / β별 평균 ρ 0.109~0.545 / 33/44종목)를 [1], [2](a), [3] 첫 줄과 대조할 것.')
print('  [2](b)(c)가 (a)와 크게 다르면, 4.4절은 (b) 또는 (c) 기준으로 다시 서술한다.')


## §24 노출 맞춤 벤치마크 — 배포 정책의 낮은 MDD는 '현금 비중' 때문인가

§23에서 배포 정책(β=−30)의 테스트 평균 ρ가 0.176으로 나왔다 — 평균적으로 자산의 약 80%를 현금으로 들고 있다. 그렇다면 등가중(100% 투자) 대비 MDD 개선(−7.43% vs −40.09%)은 대부분 노출 차이로 설명될 수 있다. 같은 노출을 준 등가중과 비교해 정책의 기여를 나눈다.

| 벤치마크 | 의미 |
|---|---|
| 등가중 100% | 기존 비교 대상 |
| 등가중 × 고정 노출 | 정책의 그 구간 평균 실현 노출만큼만 등가중에 투자, 나머지 현금 |
| 등가중 × 정책 노출 경로 | 매 스텝 정책과 **같은 노출**을 등가중으로 보유 — 정책과의 차이 = 종목 선택 효과 |

정책과 '노출 경로' 벤치마크의 차이는 **종목 선택**, '노출 경로'와 '고정 노출'의 차이는 **노출 타이밍**의 기여다. 모든 벤치마크는 같은 환경(`step_with_weights`, 비용·drift 회계 동일)으로 계산한다.

**실행**: 위 셀을 모두 실행한 뒤 이 셀만 실행(학습 없음, CPU 약 5분). 결과: `data/crypto/crypto_hrl_v4_exposure_matched.csv`

In [ ]:
# §24 — 노출 맞춤 벤치마크 (β=-30 배포 정책, α=0.01)
ALPHA_DEPLOY, BETA_DEPLOY = 0.01, -30
if 'valid_buckets' not in globals():
    valid_buckets = [b for b in train_buckets if '2025-10-01' <= bucket_to_date(b) <= '2025-12-31']


def run_policy_with_exposure(beta, alpha, buckets):
    """배포 정책 백테스트(§15 run_v4_solo_smoothed_on과 동일) + 매 스텝 실제 위험자산 노출(비중 합) 기록."""
    env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=TOP_K, **env_kwargs)
    model = pool_models_v4[beta]
    obs, _ = env.reset()
    done, expo = False, []
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        target_w, rho, top_idx = env.action_to_weights(a)
        w = (1 - alpha) * env.weights + alpha * target_w
        expo.append(float(w.sum()))
        obs, _, done, _, _ = env.step_with_weights(w)
    return np.array(env.value_history), np.array(expo)


def run_ew_with_exposure(buckets, exposure):
    """거래 가능 종목 등가중 × 노출(스칼라 또는 스텝별 배열), 나머지는 현금. 같은 환경 회계 사용."""
    env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=TOP_K, **env_kwargs)
    env.reset()
    done, t = False, 0
    while not done:
        e = exposure if np.isscalar(exposure) else exposure[min(t, len(exposure) - 1)]
        ew = _mask_untradable(np.ones(env.n_assets, dtype=np.float32), env._tradable(env.buckets[env.t]))
        _, _, done, _, _ = env.step_with_weights((e * ew).astype(np.float32))
        t += 1
    return np.array(env.value_history)


def _perf24(net, steps_per_year=48 * 365):
    rets = np.diff(net) / net[:-1]
    return {'총 수익률': net[-1] - 1,
            'Sharpe': rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year),
            'MDD': float(((net / np.maximum.accumulate(net)) - 1).min()),
            '연환산 변동성': rets.std() * np.sqrt(steps_per_year)}


rows = []
for split, buckets in [('valid', valid_buckets), ('test', test_buckets)]:
    pol_net, expo = run_policy_with_exposure(BETA_DEPLOY, ALPHA_DEPLOY, buckets)
    curves = {
        f'β={BETA_DEPLOY} 배포 정책': pol_net,
        '등가중 100%': run_ew_with_exposure(buckets, 1.0),
        '등가중 × 고정 노출': run_ew_with_exposure(buckets, float(expo.mean())),
        '등가중 × 정책 노출 경로': run_ew_with_exposure(buckets, expo),
    }
    print(f'[{split}] 정책 실현 노출 — 평균 {expo.mean():.3f} | 중앙값 {np.median(expo):.3f} | '
          f'처음 1일 평균 {expo[:48].mean():.3f} | 마지막 1일 평균 {expo[-48:].mean():.3f}')
    for k, net in curves.items():
        rows.append({'split': split, 'strategy': k, 'mean_exposure': expo.mean() if '노출' in k or '정책' in k else 1.0,
                     **_perf24(net)})

expo_df = pd.DataFrame(rows)
fmt = expo_df.copy()
for c in ['총 수익률', 'MDD', '연환산 변동성', 'mean_exposure']:
    fmt[c] = (fmt[c] * 100).map('{:.2f}%'.format)
fmt['Sharpe'] = fmt['Sharpe'].map('{:.3f}'.format)
print(); print(fmt.to_string(index=False))

# valid/test 샤프 격차 — §4.6과 같은 진단을 벤치마크에도 적용
piv = expo_df.pivot(index='strategy', columns='split', values='Sharpe')
piv['|valid-test| 격차'] = (piv['valid'] - piv['test']).abs()
print('\n--- valid/test Sharpe 격차 ---'); print(piv.round(3).to_string())

expo_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v4_exposure_matched.csv', index=False, encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_v4_exposure_matched.csv')

### 판정 기준

- **'등가중 × 노출 경로'의 MDD가 정책과 비슷하다** → MDD 개선은 노출을 낮춘 효과. 논문 기여 4번의 "MDD −40.09% → −7.43%(82% 감소)"는 노출 맞춤 벤치마크 대비 수치로 바꾸고, 평균 노출(약 18%)을 명시한다.
- **정책의 Sharpe가 '등가중 × 노출 경로'보다 높다** → 같은 노출에서 종목 선택이 기여한 것. 다만 §17 top-K 스윕에서 확신도 IC≈0이었으니, 차이가 작으면 "종목 선택 기여 없음"과 일관된다.
- **'노출 경로'가 '고정 노출'보다 낫다** → 노출을 언제 늘리고 줄이는지(타이밍)가 기여한 것.
- 참고: 등가중에 고정 비율을 곱하면 Sharpe는 거의 그대로이고(현금 수익률 0) MDD와 변동성만 줄어든다. 그래서 Sharpe 비교의 핵심은 '노출 경로' 벤치마크다.
- **α=0.01 스무딩 때문에 정책은 100% 투자 상태(초기 등가중)에서 천천히 내려온다.** 처음 1일 평균 노출이 크게 높으면, 초기 구간이 성과에 미친 영향도 함께 적는다.

## §25 — 저턴오버 아티팩트 진단 (원본 §13-F-보조, v4 라우터+풀 α=0.01)

In [ ]:
# §25 — v4 라우터+풀 + α=0.01 관성 스무딩 테스트 실행 후 전반부/후반부 수익·턴오버 추이·종목 회전 진단
ALPHA_V4 = 0.01

def run_v4_smoothed(alpha):
    env = MacroRouterEnvV4(
        buckets=test_buckets, pool_models=[pool_models_v4[b] for b in POOL_BETAS],
        route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, top_k=TOP_K, **env_kwargs,
    )
    obs, _ = env.reset()
    inner = env.inner
    done = False
    while not done:
        r_action, _ = router_v4.predict(obs, deterministic=True)
        model = env.pool_models[int(r_action)]
        done_inner = False
        for _ in range(env.route_buckets):
            a, _ = model.predict(inner._get_obs(), deterministic=True)
            target_w, rho, top_idx = inner.action_to_weights(a)
            w = (1 - alpha) * inner.weights + alpha * target_w
            _, _, done_inner, _, info = inner.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
            env._nav_history.append(info['portfolio_value'])
            if done_inner:
                break
        obs = env._macro_obs()
        done = done_inner or inner.t >= len(inner.buckets) - 1
    return inner

diag_env = run_v4_smoothed(ALPHA_V4)
turns  = np.array(diag_env.turnover_history)
values = np.array(diag_env.value_history)
n = len(turns)
decile = max(n // 10, 1)
half = n // 2
first_half_ret  = values[half] / values[0] - 1
second_half_ret = values[-1]  / values[half] - 1
sel = diag_env.selection_history
q = max(n // 4, 1)
first_q_assets = set(a for s in sel[:q] for a in s)
last_q_assets  = set(a for s in sel[-q:] for a in s)
jaccard = len(first_q_assets & last_q_assets) / len(first_q_assets | last_q_assets)

print(f'v4 라우터+풀 α={ALPHA_V4} 총수익률 {(values[-1]-1)*100:.2f}% (n={n})')
print(f'턴오버 평균 {turns.mean()*100:.4f}% | 앞10% {turns[:decile].mean()*100:.4f}% | 뒤10% {turns[-decile:].mean()*100:.4f}% | '
      f'뒤10% 중 턴오버=0 비율 {(turns[-decile:] == 0).mean()*100:.1f}%')
print(f'전반부 수익률 {first_half_ret*100:.2f}% | 후반부 수익률 {second_half_ret*100:.2f}%')
print(f'앞 1/4 선택 종목 {len(first_q_assets)}개 | 뒤 1/4 {len(last_q_assets)}개 | 자카드 {jaccard:.2f}')

pd.DataFrame([{
    'alpha': ALPHA_V4, 'total_return': values[-1] - 1, 'turnover_mean': turns.mean(),
    'turnover_first10': turns[:decile].mean(), 'turnover_last10': turns[-decile:].mean(),
    'last10_zero_turnover_frac': (turns[-decile:] == 0).mean(),
    'first_half_return': first_half_ret, 'second_half_return': second_half_ret,
    'first_q_assets': len(first_q_assets), 'last_q_assets': len(last_q_assets), 'jaccard': jaccard,
}]).to_csv(CRYPTO_DIR / 'crypto_hrl_v4_static_artifact_diag.csv', index=False, encoding='utf-8-sig')
print('저장 완료: crypto_hrl_v4_static_artifact_diag.csv')


## §19 — valid/test 샤프 격차 블록 순열검정 (6.10절)

In [ ]:
# §19-A — 블록 순열검정: valid/test 샤프 격차가 우연(구간 교환가능성)만으로 설명되는가
STEPS_PER_YEAR = 48 * 365
BLOCK_LEN = 48  # 1일치 스텝 수(30분봉)

# run_v4_solo_smoothed_on은 원래 §15(셀80)에서 정의되지만, 그 셀은 BEST_ALPHA_V4_VALID·
# BEST_ALPHA_V4_MATCHED(§13-I/§14 체인 전체)까지 있어야 끝까지 실행되는 무거운 셀이다 —
# 이 함수 정의 자체는 그 체인과 무관하므로, §18과 마찬가지로 여기서 idempotent하게 로컬
# 재정의해 §19를 §18 이후 가벼운 사전 셀만으로 독립 실행 가능하게 만든다.
if 'run_v4_solo_smoothed_on' not in globals():
    def run_v4_solo_smoothed_on(beta, alpha, buckets):
        env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=TOP_K, **env_kwargs)
        model = pool_models_v4[beta]
        obs, _ = env.reset()
        gross, g = [1.0], 1.0
        done = False
        while not done:
            a, _ = model.predict(obs, deterministic=True)
            target_w, rho, top_idx = env.action_to_weights(a)
            w = (1 - alpha) * env.weights + alpha * target_w
            obs, _, done, _, info = env.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
            g *= 1 + info['port_ret']
            gross.append(g)
        return np.array(env.value_history), np.array(gross), list(env.turnover_history)
    print('[§19] run_v4_solo_smoothed_on 로컬 재정의(§15 셀 미실행 세션 대비)')

if 'ALPHA_DEPLOY' not in globals():
    ALPHA_DEPLOY = 0.01
    print('[§19] ALPHA_DEPLOY=0.01 로컬 정의(§18 셀 미실행 세션 대비)')


def sharpe_of(rets, steps_per_year=STEPS_PER_YEAR):
    return rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year)


def block_gap_permutation_test(valid_net, test_net, block_len=BLOCK_LEN, n_perm=2000, seed=0):
    """H0: valid 구간과 test 구간의 수익률 블록은 교환 가능하다(그 정책의 성과가 두 구간에서
    구조적으로 다르지 않다). 관측된 |valid Sharpe - test Sharpe|가 풀링 후 무작위 블록
    재배정으로도 흔히 나오는 크기인지 확인한다. p-value = P(|permuted gap| >= |observed gap|)."""
    rets_v = np.diff(valid_net) / valid_net[:-1]
    rets_t = np.diff(test_net) / test_net[:-1]
    observed_gap = sharpe_of(rets_t) - sharpe_of(rets_v)

    pooled = np.concatenate([rets_v, rets_t])
    blocks = [pooled[i:i + block_len] for i in range(0, len(pooled), block_len)]
    n_blocks_v = int(round(len(rets_v) / block_len))

    rng = np.random.default_rng(seed)
    idx = np.arange(len(blocks))
    perm_gaps = np.empty(n_perm)
    for i in range(n_perm):
        rng.shuffle(idx)
        perm_v = np.concatenate([blocks[j] for j in idx[:n_blocks_v]])
        perm_t = np.concatenate([blocks[j] for j in idx[n_blocks_v:]])
        perm_gaps[i] = sharpe_of(perm_t) - sharpe_of(perm_v)

    p_value = (np.sum(np.abs(perm_gaps) >= abs(observed_gap)) + 1) / (n_perm + 1)
    return observed_gap, perm_gaps, p_value


def block_bootstrap_ci(net, block_len=BLOCK_LEN, n_boot=2000, seed=0, alpha_ci=0.05):
    """단일 구간 Sharpe의 블록부트스트랩 95% CI(참고용 — 순열검정의 보조 진단)."""
    rets = np.diff(net) / net[:-1]
    n = len(rets)
    n_blocks = int(np.ceil(n / block_len))
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot)
    for b in range(n_boot):
        starts = rng.integers(0, max(n - block_len, 1) + 1, size=n_blocks)
        sample = np.concatenate([rets[s:s + block_len] for s in starts])[:n]
        boot[b] = sharpe_of(sample)
    lo, hi = np.quantile(boot, [alpha_ci / 2, 1 - alpha_ci / 2])
    return lo, hi


print('=== [§19] β별 valid/test 샤프 격차 — 블록 순열검정 (n_perm=2000, block=1일) ===\n')
gap_test_rows = {}
for beta in POOL_BETAS:
    v_net, _, _ = run_v4_solo_smoothed_on(beta, ALPHA_DEPLOY, valid_buckets)
    t_net, _, _ = run_v4_solo_smoothed_on(beta, ALPHA_DEPLOY, test_buckets)
    observed_gap, perm_gaps, p_value = block_gap_permutation_test(v_net, t_net)
    v_lo, v_hi = block_bootstrap_ci(v_net)
    t_lo, t_hi = block_bootstrap_ci(t_net)
    label = f'β={beta}' + (' (배포)' if beta == -30 else '')
    gap_test_rows[label] = {
        'valid Sharpe': sharpe_of(np.diff(v_net) / v_net[:-1]),
        'valid 95% CI': f'[{v_lo:.3f}, {v_hi:.3f}]',
        'test Sharpe': sharpe_of(np.diff(t_net) / t_net[:-1]),
        'test 95% CI': f'[{t_lo:.3f}, {t_hi:.3f}]',
        'gap(test-valid)': observed_gap,
        'permutation p-value': p_value,
    }
    print(f'{label}: gap={observed_gap:+.3f}, p={p_value:.4f}')

gap_test_df = pd.DataFrame(gap_test_rows).T
print()
print(gap_test_df.to_string())
gap_test_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v4_gap_permutation_test.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_v4_gap_permutation_test.csv')